In [1]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
import pandas as pd
import numpy as np
from itertools import product
import matplotlib.pyplot as plt
import time

In [ ]:
X_test = pd.read_csv('X_test.csv')
X_train_full = pd.read_csv('X_train.csv')
y_train_full = pd.read_csv('y_train.csv')

# Para manejar el tema del overfitting luego, ya del principio spliteo la data

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.20, random_state=42
)

In [3]:
#Antes que nada, un codigo general para chequear distintas variables de una 
#Pongo lo de timear para algo q voy a probar dsp

start_time = time.perf_counter()

selected_features = ['fage', 'mage']

X_tr_test = pd.get_dummies(X_tr[selected_features], drop_first=True) #mini dataset con las features q elegí, y get_dummies hace eso de poner 0 o 1 a strings. 
X_val_test = pd.get_dummies(X_val[selected_features], drop_first=True) #lo mismo para el validation set


model = LinearRegression()
model.fit(X_tr_test, y_tr)

#O sea, a mi me parece mas logico que si yo entreno con un set de features, el mae y el r2 se calculen sobre lo que dejé, onda lo que 
#no es de training

predictions = model.predict(X_val_test)
mae = mean_absolute_error(y_val, predictions)
r2 = r2_score(y_val, predictions)

end_time = time.perf_counter()
execution_time = end_time - start_time

print(f"Features: {selected_features}")
print(f"MAE: {mae:.4f}")
print(f"r²: {r2:.4f}")
print(f"Execution time: {execution_time:.4f} seconds")

Features: ['fage', 'mage']
MAE: 0.8358
r²: -0.0282
Execution time: 0.0599 seconds


In [ ]:
#ok. Tarda 20 milisegundos puedo simplemente ver que combinacion de las 2^9-1 que hay da menor MAE son tipo 10s de ejecucion (en mi pc)

# Voy a definir un par de helper functions para hacer eso

def filter(features, X_fit, X_other): #Me conviene tener esto porque siempre voy a tener que hacer lo mismo para tr y val. 
    """
    Transforma strings en "True/False" y filtra arrays dado columnas
    """
    X_fit_filtered = pd.get_dummies(X_fit[list(features)], drop_first=True)
    X_other_filtered = pd.get_dummies(X_other[list(features)], drop_first=True)
    X_other_filtered = X_other_filtered.reindex(columns=X_fit_filtered.columns, fill_value=0)
    return X_fit_filtered, X_other_filtered


def evaluate_features(features, X_tr, y_tr, X_val, y_val):
    """
    Fitea una lineal dado unas features al X_tr y y_tr, y calcula el mae y r2 en el X_val y y_val
    """
    X_tr_enc, X_val_enc = filter(features, X_tr, X_val)

    model = LinearRegression()
    model.fit(X_tr_enc, y_tr)
    preds_val = model.predict(X_val_enc)

    mae = mean_absolute_error(y_val, preds_val)
    r2 = r2_score(y_val, preds_val)
    return mae, r2

In [ ]:
# #sanitycheck
selected_features = ['fage', 'mage']
mae, r2 = evaluate_features(selected_features, X_tr, y_tr, X_val, y_val)


print(f"Features: {selected_features}")
print(f"MAE: {mae:.4f}")
print(f"r2: {r2:.4f}")
print(f"Tiempo de ejecución: {execution_time:.4f} seconds")

Features: ['fage', 'mage']
Held-out MAE: 0.8358
Held-out r2: -0.0282
Execution time: 0.0599 seconds


# Fuerza bruta con linear regression

Bueno ahora si, voy a hacer fuerza bruta sobre todas las posibles combinaciones de features

In [ ]:
def evaluate_all_feature_subsets(X_tr, y_tr, X_val, y_val):
    '''
    Evalúa todos los posibles subconjuntos de features y devuelve un DataFrame con los resultados ordenados por MAE.
    '''
    
    all_features = np.array(['fage', 'mage', 'mature', 'visits', 'gained', 'sex', 'habit', 'marital', 'whitemom'])
    n = len(all_features)
    masks = np.array(list(product([0, 1], repeat=n)))[1:]  # esto es una mascara para poder agarrar todas las 2^n-1 combinaciones de all_features
    results = []
    for mask in masks:
        feats = all_features[mask.astype(bool)]
        mae, r2 = evaluate_features(feats, X_tr, y_tr, X_val, y_val)
        results.append({'features': feats, 'mae': mae, 'r2': r2})
    return pd.DataFrame(results).sort_values(by='mae').reset_index(drop=True)


results_df = evaluate_all_feature_subsets(X_tr, y_tr, X_val, y_val)

print("Top 10 combos:")
print(results_df.head(10))

Top 10 combos:
                                          features       mae        r2
0          [fage, mage, visits, marital, whitemom]  0.825491  0.002360
1                   [fage, mage, visits, whitemom]  0.825730  0.002219
2                [mage, visits, marital, whitemom]  0.825792  0.004061
3                         [mage, visits, whitemom]  0.825983  0.003781
4                 [mage, visits, gained, whitemom]  0.826831 -0.023562
5           [fage, mage, visits, gained, whitemom]  0.826853 -0.025113
6        [mage, visits, gained, marital, whitemom]  0.826864 -0.023151
7  [fage, mage, visits, gained, marital, whitemom]  0.826907 -0.024907
8                  [fage, mage, marital, whitemom]  0.827700  0.000273
9                        [mage, marital, whitemom]  0.828013  0.001067


# Probando otros split

El codigo de arriba esta bueno, pero el problema con el codigo es que _qué pasa si justo la combinacion de features que elegí le gusta al split especifico que generó la seed_? es decir, si elijo otra seed, la mejor combinacion es otra?

Bueno para resolver esto voy a generar muchos splits distintos y ver que combinacion de features se mantiene mas seguido en el top 3 o top 10 o lo que sea, la idea es que si una combo de features es siempre muy buena, entonces tiene sentido suponer que no estoy overfiteando y efectivamente es la combinación que mas se adapta a la señal. 

Ojo que esto si tarda mucho, es maso 8-10s cada split, asi que si hago 10 splits distintos ya es casi 2 minutos, ademas no esta optimizado. Más abajo le pedí a Claued que lo haga mas performante pero la lógica es la misma solo que lo paralelizó todo. Esta celda de acá abajo se puede saltear

In [ ]:
#Esto si que tarda eh! bajar n_repeats asi tarda menos xD  VER ABAJO!!


#Bueno para que sea justo, la idea es correr la funcion del principio train_test_split muchas veces pero con distintas seeds O sea genero
# un numero aleatorio de seeds xD. 
def get_best_combinations_repeated(X_data, y_data, n_repeats, split_frac=0.20, seed=42):
    '''
    Repite Evaluate_all_feature_subsets varias veces con distintas seeds y devuelve un DataFrame con los resultados de todas las corridas.

    Notar que los inputs de esto son los datos completos
    '''

    rng = np.random.default_rng(seed) # uso la seed que ya tenía antes para generar muchas otras seeds. 

    all_run_results = []

    for run in range(n_repeats):
        run_seed = int(rng.integers(0, 1e6))

        X_tr_r, X_val_r, y_tr_r, y_val_r = train_test_split(
            X_data, y_data, test_size=split_frac, random_state=run_seed
        )

        run_df = evaluate_all_feature_subsets(X_tr_r, y_tr_r, X_val_r, y_val_r)
        run_df['features'] = run_df['features'].apply(lambda f: ", ".join(f))
        run_df['run'] = run
        all_run_results.append(run_df)

    return pd.concat(all_run_results, ignore_index=True)

repeated_results = get_best_combinations_repeated(X_train_full, y_train_full, n_repeats=50)


In [24]:
top_per_run = repeated_results.groupby('run', group_keys=False).apply(lambda d: d.nsmallest(10, 'mae'))
top_counts = top_per_run['features'].value_counts()

print("Combo de features que mas veces apareció en el top 5")
print(top_counts.head(10))

Combo de features que mas veces apareció en el top 5
features
fage, mature, gained, sex, habit, whitemom             10
fage, gained, sex, habit, whitemom                      8
mature, gained, sex, habit, whitemom                    7
fage, gained, sex, habit, marital, whitemom             7
mage, mature, gained, sex, habit, whitemom              6
fage, mature, visits, gained, sex, whitemom             6
mature, gained, sex, whitemom                           6
mage, visits, gained, sex, whitemom                     6
fage, visits, gained, sex, habit, marital, whitemom     6
fage, mature, visits, gained, sex, habit, whitemom      6
Name: count, dtype: int64


Interesante esto, antes hicimos el codigo probando mae y r2 contra el set de entrenamiento completo, o sea nunca spliteando.

y llegaba a: 
 ['fage', 'gained', 'sex', 'habit', 'whitemom'] 
como ganador absoluto

Ahora salio 

[fage, mature, gained, sex, habit, whitemom]    como ganador con 10 victorias en n=50

y [fage, gained, sex, habit, whitemom] con 8. 

Para validar esto me gustaría probar con mas splits todavía a ver si hay un claro ganador antes mas seeds

In [14]:
#Para ver los numeros
print(evaluate_features(['fage','gained','sex','habit','whitemom'], X_tr, y_tr, X_val, y_val))

print(evaluate_features(['fage','mature','gained','sex','habit','whitemom'], X_tr, y_tr, X_val, y_val))

(0.8569527057081225, -0.030831005358918695)
(0.8593265964934012, -0.032123092917161644)


Bue qcyo no es una locura. Supuestamente a Chatgpt pro le dio MAE = 1.07 Y R2 =0.099 sobre el y_test (que yo no tengo) asi que bueno, por lo menos para una seed le ganamos a chatgpt

In [ ]:
#Optimización:

#Ok este codigo no es mío, Simplemente le mandé mi codigo a claude y le dije "hacelo lo mas rapido q se pueda"

#No tengo ni idea de que pasa acá pero es rapidisimo! 

import numpy as np, pandas as pd, time
from itertools import product
from sklearn.model_selection import train_test_split

def get_best_combinations_repeated(X_data, y_data, n_repeats, split_frac=0.20, seed=42,
        all_features=('fage','mage','mature','visits','gained','sex','habit','marital','whitemom')):
    all_features = list(all_features)

    blocks, feat_cols, p = [], [], 0
    for f in all_features:
        d = pd.get_dummies(X_data[[f]], drop_first=True)
        blocks.append(np.asarray(d, dtype=np.float64))
        feat_cols.append(np.arange(1 + p, 1 + p + d.shape[1])); p += d.shape[1]
    n = len(X_data)
    Xa = np.empty((n, p + 1)); Xa[:, 0] = 1.0; Xa[:, 1:] = np.hstack(blocks)
    mu, sd = Xa[:, 1:].mean(0), Xa[:, 1:].std(0); sd[sd == 0] = 1.0
    Xa[:, 1:] = (Xa[:, 1:] - mu) / sd            
    y = np.asarray(y_data, dtype=np.float64).ravel()
    assert np.isfinite(Xa).all() and np.isfinite(y).all(), "hay NaNs en los datos"

    masks = np.array(list(product([0, 1], repeat=len(all_features))), dtype=bool)[1:]
    names = np.array([", ".join([f for f, m in zip(all_features, k) if m]) for k in masks], dtype=object)
    groups = {}
    for i, msk in enumerate(masks):
        cols = np.concatenate([[0]] + [feat_cols[j] for j in np.flatnonzero(msk)])
        g = groups.setdefault(len(cols), ([], [])); g[0].append(i); g[1].append(cols)
    groups = {k: (np.array(a), np.array(b)) for k, (a, b) in groups.items()}

    idx, S = np.arange(n), len(masks)
    rng = np.random.default_rng(seed)
    mae_out, r2_out = np.empty((n_repeats, S)), np.empty((n_repeats, S))

    for run in range(n_repeats):
        run_seed = int(rng.integers(0, 1e6))
        itr, iva = train_test_split(idx, test_size=split_frac, random_state=run_seed)
        Xtr, Xva, ytr, yva = Xa[itr], Xa[iva], y[itr], y[iva]
        G, b = Xtr.T @ Xtr, Xtr.T @ ytr
        ss_tot = np.sum((yva - yva.mean()) ** 2)
        for _, (rows, C) in groups.items():
            Gs, bs = G[C[:, :, None], C[:, None, :]], b[C]
            try:
                beta = np.linalg.solve(Gs, bs[:, :, None])[:, :, 0]
            except np.linalg.LinAlgError:
                beta = (np.linalg.pinv(Gs) @ bs[:, :, None])[:, :, 0]
            B = np.zeros((len(rows), Xa.shape[1])); np.put_along_axis(B, C, beta, axis=1)
            res = B @ Xva.T - yva                       # (subsets, n_val) at once
            mae_out[run, rows] = np.abs(res).mean(1)
            r2_out[run, rows] = 1.0 - (res * res).sum(1) / ss_tot

    order = np.argsort(mae_out, axis=1, kind='stable')   
    return pd.DataFrame({'features': np.concatenate([names[o] for o in order]),
                         'mae': np.take_along_axis(mae_out, order, 1).ravel(),
                         'r2':  np.take_along_axis(r2_out,  order, 1).ravel(),
                         'run': np.repeat(np.arange(n_repeats), S)})

t = time.perf_counter()
repeated_results_fast = get_best_combinations_repeated(X_train_full, y_train_full, n_repeats=10000)
print(f"listo en {time.perf_counter()-t:.3f} s -> {len(repeated_results_fast)} filas")

listo en 28.378 s -> 5110000 filas


In [19]:
top_per_run = repeated_results_fast.groupby('run', group_keys=False).apply(lambda d: d.nsmallest(1, 'mae'))
top_counts = top_per_run['features'].value_counts()

In [20]:
print(top_counts.head(10))

features
fage, gained, sex, habit, whitemom                    299
fage, visits, gained, sex, habit, whitemom            159
fage, mature, gained, sex, habit, whitemom            155
fage, gained, sex, habit, marital, whitemom           154
gained, sex, habit, whitemom                          147
fage, gained, sex, whitemom                           145
mage, gained, sex, habit, whitemom                    141
gained, sex, whitemom                                 133
fage, mature, visits, gained, sex, habit, whitemom    128
mature, visits, gained, sex, habit, whitemom          119
Name: count, dtype: int64


bueno despues de hacer esto diez mil veces gana fage gained sex habit whitemom, lo mismo que encontramos antes. 

Entonces, por lo menos en regresión lineal, cerramos en un ajuste de 5 variables con 
- La edad del padre
- El peso que aumentó la madre
- El sexo del bebé
- Si la madre es fumadora o no 
- Si la mamá es de tez blanca o no

y nos dió $MAE = 0.8569527057081225$ y $R^2= -0.030831005358918695$ tomando el 20% de los datos en donde no se entrenó el modelo.


# Ridge/Lasso

In [25]:
def evaluate_features_reg(features, X_tr, y_tr, X_val, y_val, alpha, model_type='ridge'):
    X_tr_filtered, X_val_filtered = filter(features, X_tr, X_val)

    model = Ridge(alpha=alpha) if model_type == 'ridge' else Lasso(alpha=alpha)
    model.fit(X_tr_filtered, y_tr)
    preds_val = model.predict(X_val_filtered)

    mae = mean_absolute_error(y_val, preds_val)
    r2 = r2_score(y_val, preds_val)
    return mae, r2


selected_features = ['fage','mature', 'gained', 'sex', 'habit', 'whitemom']

for alpha in [0.01, 0.1, 1, 10, 100]:
    mae_r, r2_r = evaluate_features_reg(selected_features, X_tr, y_tr, X_val, y_val, alpha, 'ridge')
    mae_l, r2_l = evaluate_features_reg(selected_features, X_tr, y_tr, X_val, y_val, alpha, 'lasso')
    print(f"alpha={alpha}: Ridge MAE={mae_r:.4f} R2={r2_r:.4f} | Lasso MAE={mae_l:.4f} R2={r2_l:.4f}")

alpha=0.01: Ridge MAE=0.8593 R2=-0.0321 | Lasso MAE=0.8529 R2=-0.0197
alpha=0.1: Ridge MAE=0.8593 R2=-0.0320 | Lasso MAE=0.8462 R2=-0.0535
alpha=1: Ridge MAE=0.8586 R2=-0.0305 | Lasso MAE=0.8440 R2=-0.0217
alpha=10: Ridge MAE=0.8535 R2=-0.0213 | Lasso MAE=0.8431 R2=-0.0166
alpha=100: Ridge MAE=0.8433 R2=-0.0194 | Lasso MAE=0.8431 R2=-0.0166


# polinomios n degree

In [26]:
def evaluate_features_poly(features, X_tr, y_tr, X_val, y_val, degree):
    X_tr_filtered, X_val_filtered = filter(features, X_tr, X_val)

    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_tr_poly = poly.fit_transform(X_tr_filtered)
    X_val_poly = poly.transform(X_val_filtered)   

    model = LinearRegression()
    model.fit(X_tr_poly, y_tr)
    preds_val = model.predict(X_val_poly)

    mae = mean_absolute_error(y_val, preds_val)
    r2 = r2_score(y_val, preds_val)
    return mae, r2


selected_features = ['fage', 'mature','gained', 'sex', 'habit', 'whitemom']

for degree in [1, 2, 3, 4, 5]:
    mae, r2 = evaluate_features_poly(selected_features, X_tr, y_tr, X_val, y_val, degree)
    print(f"Degree {degree}: MAE={mae:.4f} R2={r2:.4f}")

Degree 1: MAE=0.8593 R2=-0.0321
Degree 2: MAE=0.8655 R2=-0.0431
Degree 3: MAE=0.8898 R2=-0.1686
Degree 4: MAE=0.9840 R2=-0.7483
Degree 5: MAE=1.0866 R2=-2.3759


Tampoco parece mejorar muchísimo el modelo con un ajuste ridge/lasso o polinómico. Dan casi siempre el mismo Mae y R^2. 

# Terminos de interaccion

Tomi probó por su cuenta como afectan los términos de interaccion, llegó a que ['mature','gained','sex','habit','whitemom'] con interaccion entre ['mage','visits'] funciona mejor.

La idea es ver si la combinacion a la que llegamos antes ['fage','gained','sex','habit','whitemom'] es mejor o peor que la de interacción. 

In [28]:
n_splits = 100

features_non_interact = ['fage', 'gained', 'sex', 'habit', 'whitemom']
features_base_interact = ['mature', 'gained', 'sex', 'habit', 'whitemom', 'mage', 'visits']

mae_non = []
mae_int = []

r2_non = []
r2_int = []

rng = np.random.default_rng(99) # Numero random para probar distintas seeds 

for i in range(n_splits):
    run_seed = int(rng.integers(0, 1e6))
    
    X_tr_split, X_val_split, y_tr_split, y_val_split = train_test_split(
        X_train_full, y_train_full, test_size=0.20, random_state=run_seed
    )
    
    X_tr_non, X_val_non = filter(features_non_interact, X_tr_split, X_val_split)
    
    model_non = LinearRegression()
    model_non.fit(X_tr_non, y_tr_split)
    preds_non = model_non.predict(X_val_non)
    mae_non.append(mean_absolute_error(y_val_split, preds_non))
    r2_non.append(r2_score(y_val_split, preds_non))
    
    X_tr_int, X_val_int = filter(features_base_interact, X_tr_split, X_val_split)
    
    X_tr_int['mage_x_visits'] = X_tr_int['mage'] * X_tr_int['visits']
    X_val_int['mage_x_visits'] = X_val_int['mage'] * X_val_int['visits']
    
    X_tr_int = X_tr_int.drop(columns=['mage', 'visits'])
    X_val_int = X_val_int.drop(columns=['mage', 'visits'])
    
    model_int = LinearRegression()
    model_int.fit(X_tr_int, y_tr_split)
    preds_int = model_int.predict(X_val_int)
    mae_int.append(mean_absolute_error(y_val_split, preds_int))
    r2_int.append(r2_score(y_val_split, preds_int))

results_df = pd.DataFrame({
    'Split': range(1, n_splits + 1),
    'MAE_Non_Interact': mae_non,
    'MAE_Interact': mae_int,
    'R2_Non_Interact': r2_non,
    'R2_Interact': r2_int
})

print(results_df.to_string(index=False))
print(f"\nMean MAE (Non-Interaction): {np.mean(mae_non):.4f}")
print(f"Mean MAE (Interaction): {np.mean(mae_int):.4f}")
print(f"Mean R2 (Non-Interaction): {np.mean(r2_non):.4f}")
print(f"Mean R2 (Interaction): {np.mean(r2_int):.4f}")

 Split  MAE_Non_Interact  MAE_Interact  R2_Non_Interact  R2_Interact
     1          0.807946      0.806041        -0.085427    -0.080740
     2          0.876292      0.898108        -0.023390    -0.058042
     3          0.939564      0.955414         0.045075     0.027042
     4          0.870863      0.893991         0.014885    -0.022153
     5          0.912748      0.927246         0.024095     0.004903
     6          0.879323      0.877441        -0.000033     0.016715
     7          0.900560      0.905292         0.105495     0.102366
     8          0.889584      0.897086         0.084590     0.071240
     9          0.880050      0.893098         0.123052     0.102869
    10          0.884125      0.884106         0.113548     0.113279
    11          0.840118      0.840164         0.020890     0.021201
    12          0.919969      0.918866         0.035624     0.041407
    13          0.863639      0.863415         0.098638     0.101747
    14          0.856646      0.86

Practicamente no hay diferencia o gana por muy poquito el ajuste sin interacción.

# Respuesta final

Nos decidimos quedar con el ajuste lineal de 5 variables mencionado previamente

In [31]:
final_features = ['fage', 'gained', 'sex', 'habit', 'whitemom']

X_train_final, X_test_final = filter(final_features, X_train_full, X_test)

final_model = LinearRegression()
final_model.fit(X_train_final, y_train_full)

y_pred_test = final_model.predict(X_test_final)

submission = pd.DataFrame({'y_pred': y_pred_test.ravel()})
submission.to_csv('predictions.csv', index=False)